# 02 — Iceberg via REST catalog

The `iceberg` catalog points at the `tabular/iceberg-rest` service on :8181, which stores data on `s3a://spark-warehouse/iceberg/` (MinIO). Namespace `demo` is created at stack startup.

In [ ]:
from spark_session import get_spark
spark = get_spark("02-iceberg")

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.demo")
spark.sql("DROP TABLE IF EXISTS iceberg.demo.events")
spark.sql("""
CREATE TABLE iceberg.demo.events (
    id     BIGINT,
    ts     TIMESTAMP,
    kind   STRING,
    amount DOUBLE
) USING iceberg
PARTITIONED BY (days(ts))
""")

In [ ]:
spark.sql("""
INSERT INTO iceberg.demo.events VALUES
    (1, TIMESTAMP '2026-06-01 10:15:00', 'click', 1.0),
    (2, TIMESTAMP '2026-06-01 10:16:00', 'view',  0.0),
    (3, TIMESTAMP '2026-06-02 08:01:00', 'purchase', 42.50)
""")
spark.table("iceberg.demo.events").show()

In [ ]:
# A second batch — gives us a second snapshot.
from pyspark.sql import Row
from datetime import datetime
more = spark.createDataFrame([
    Row(id=4, ts=datetime(2026, 6, 2, 9, 0), kind="click",   amount=0.0),
    Row(id=5, ts=datetime(2026, 6, 3, 9, 0), kind="refund", amount=-7.25),
])
more.writeTo("iceberg.demo.events").append()
spark.sql("SELECT * FROM iceberg.demo.events ORDER BY ts").show()

In [ ]:
# Snapshot history exposed by Iceberg's metadata tables.
spark.sql("SELECT committed_at, snapshot_id, operation FROM iceberg.demo.events.snapshots").show(truncate=False)

In [ ]:
spark.sql("SELECT file_path, record_count, file_size_in_bytes FROM iceberg.demo.events.files").show(truncate=False)

In [ ]:
# Release the SparkSession so cluster cores free up and the History Server can ingest this app.
spark.stop()